In [12]:
# Default values - Papermill will replace these
asset = "USDT"
fiat = "VES"

In [18]:
import requests
import scrapbook as sb
from datetime import datetime

# 1. API Configuration
url = "https://p2p.binance.com/bapi/c2c/v2/friendly/c2c/adv/search"
payload = {
    "asset": asset,
    "fiat": fiat,
    "merchantCheck": False,
    "page": 1,
    "rows": 20,
    "tradeType": "BUY"
}

try:
    # 2. Get Data
    response = requests.post(url, json=payload)
    data = response.json()
    
    # 3. Calculate Average
    precios = [float(oferta['adv']['price']) for oferta in data['data']]
    promedio = sum(precios) / len(precios)
    current_time = datetime.now().strftime("%H:%M:%S")

    # 4. "GLUE" the data (This is how the master script reads it)
    sb.glue("promedio_final", promedio)
    sb.glue("timestamp", current_time)
    
    print(f"[{current_time}] Success: {promedio} {fiat}")

except Exception as e:
    print(f"Error during execution: {e}")

[15:28:07] Success: 498.60805 VES


In [ ]:
import requests
import scrapbook as sb
from datetime import datetime

url = "https://p2p.binance.com/bapi/c2c/v2/friendly/c2c/adv/search"
payload = {
    "asset": asset,
    "fiat": fiat,
    "merchantCheck": False,
    "page": 1,
    "rows": 20,
    "tradeType": "BUY"
}

try:
    response = requests.post(url, json=payload)
    data = response.json()
    
    # 1. Capture the 5 offers as a list of dictionaries
    offers_list = []
    precios = []
    
    for oferta in data['data']:
        p = float(oferta['adv']['price'])
        n = oferta['advertiser']['nickName']
        precios.append(p)
        offers_list.append({"Advertiser": n, "Price": p})

    promedio = sum(precios) / len(precios)
    current_time = datetime.now().strftime("%H:%M:%S")

    # 2. Glue the list of offers AND the average
    sb.glue("promedio_final", promedio)
    sb.glue("top_5_offers", offers_list) 
    sb.glue("timestamp", current_time)
    
    print(f"Calculated average of {len(offers_list)} offers.")
except Exception as e:
    print(f"Error: {e}")